# 01 — Data Collection (rebuilt)

**Salary source:** `backend/data/nhl_contracts.csv` (1,045 historical contracts with signing year)

**Stats source:** MoneyPuck CSVs — situations: `all`, `5on5`, `pp`, `sh`; seasons 2019–2024

**Stat-season pairing logic:**
Each contract is paired with MoneyPuck stats from the season *before* the contract started:
`stat_season = max(2019, min(Start - 1, 2024))`
Rationale: teams sign contracts based on what a player just did, not what they'll do next year.

**Known ceiling:** ~80 unmatched contracts are ELC/prospect deals for players with no NHL stats — expected and correct. Target: 900–965 matched rows before GP filter.

In [ ]:
import os, re, unicodedata, io
import pandas as pd
import numpy as np
import requests

REPO_ROOT     = os.path.dirname(os.getcwd())
CSV_PATH      = os.path.join(REPO_ROOT, 'backend', 'data', 'nhl_contracts.csv')
HEADERS       = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}
MP_BASE       = 'https://moneypuck.com/moneypuck/playerData/seasonSummary'
SEASONS       = [str(y) for y in range(2019, 2025)]
MIN_GP_SKATER = 40
MIN_GP_GOALIE = 20   # goalies: median season is 24 GP, 40 cuts 75% of them

# MoneyPuck situation column values (NOT 'pp'/'sh' — those don't exist in the data)
SIT_ALL = 'all'
SIT_EV  = '5on5'
SIT_PP  = '5on4'   # power play: our team has 5 skaters, they have 4
SIT_PK  = '4on5'   # penalty kill: our team has 4 skaters, they have 5

def norm(name):
    """Lowercase, strip accents, remove non-alphanumeric for fuzzy matching."""
    name = unicodedata.normalize('NFD', str(name))
    name = ''.join(c for c in name if unicodedata.category(c) != 'Mn')
    return re.sub(r'[^a-z0-9 ]', '', name.lower()).strip()

# Download cache — each (season, ptype) CSV is only fetched once
_raw_csv_cache = {}

def get_raw_csv(season, ptype):
    key = (season, ptype)
    if key in _raw_csv_cache:
        return _raw_csv_cache[key]
    url = f'{MP_BASE}/{season}/regular/{ptype}.csv'
    try:
        r = requests.get(url, headers=HEADERS, timeout=30)
        r.raise_for_status()
        df = pd.read_csv(io.StringIO(r.text), low_memory=False)
        _raw_csv_cache[key] = df
        return df
    except Exception as e:
        print(f'  WARN {season}/{ptype}: {e}')
        _raw_csv_cache[key] = pd.DataFrame()
        return pd.DataFrame()

def fetch_mp(season, ptype, situation=SIT_ALL):
    df = get_raw_csv(season, ptype)
    if df.empty or 'situation' not in df.columns:
        return pd.DataFrame()
    return df[df['situation'] == situation].copy()

print('Setup complete.')

## 1. Load and clean nhl_contracts.csv

In [ ]:
raw = pd.read_csv(CSV_PATH)
raw.columns = raw.columns.str.strip()

# Rename columns to clean names
contracts = raw.rename(columns={
    'Player': 'player_name',
    'Pos': 'position',
    'Team                     Currently With': 'team',
    'Age                     At Signing': 'age_at_signing',
    'Start': 'start_year',
    'End': 'end_year',
    'Yrs': 'contract_years',
    'Value': 'total_value_raw',
    'AAV': 'aav_raw',
}).copy()

# Parse AAV to float
contracts['aav'] = (
    contracts['aav_raw']
    .str.replace(r'[\$,]', '', regex=True)
    .astype(float)
)

# Stat season: the season whose stats best reflect why the player was signed
# = the season that just finished when the contract was inked
contracts['stat_season'] = contracts['start_year'].apply(
    lambda s: max(2019, min(int(s) - 1, 2024))
)

# Clean team column
contracts['team'] = contracts['team'].str.strip()
contracts['player_name'] = contracts['player_name'].str.strip()

print(f'Contracts loaded: {len(contracts):,}')
print(f'Stat season distribution:')
print(contracts['stat_season'].value_counts().sort_index())
print()
contracts[['player_name','position','team','start_year','stat_season','aav']].head(10)

## 2. Build MoneyPuck Name → Player ID Lookup

Fetch all 6 seasons of skater + goalie data and build a comprehensive name-to-ID dictionary. This is the foundation of all name matching.

In [ ]:
# name_to_id: norm(name) -> playerId string
# id_to_info: playerId -> {name, position}
name_to_id = {}
id_to_info  = {}

def fetch_mp(season, ptype, situation='all'):
    url = f'{MP_BASE}/{season}/regular/{ptype}.csv'
    try:
        r = requests.get(url, headers=HEADERS, timeout=30)
        r.raise_for_status()
        df = pd.read_csv(io.StringIO(r.text), low_memory=False)
        return df[df['situation'] == situation].copy()
    except Exception as e:
        print(f'  WARN {season}/{ptype}: {e}')
        return pd.DataFrame()

for season in SEASONS:
    for ptype in ['skaters', 'goalies']:
        df = fetch_mp(season, ptype)
        if df.empty:
            continue
        for _, row in df.iterrows():
            pid  = str(int(row['playerId']))
            name = str(row.get('name', '')).strip()
            pos  = str(row.get('position', '')).strip()
            if name and pid:
                name_to_id[norm(name)] = pid
                id_to_info[pid] = {'name': name, 'position': pos}

print(f'Name lookup built: {len(name_to_id):,} unique player names')
print(f'Unique player IDs: {len(id_to_info):,}')

## 3. Match Contract Names to MoneyPuck Player IDs

Three-pass matching:
1. Exact normalized name match
2. Last name + first initial (handles Jr./suffix/nickname differences)
3. Manual overrides for known name differences (Zuccarello-Aasen, etc.)

In [ ]:
# Known name overrides: contracts CSV name -> MoneyPuck name
OVERRIDES = {
    # Skaters
    'mats zuccarello aasen': 'mats zuccarello',
    'arseny gritsyuk':       'arseni gritsyuk',
    'anthony deangelo':      'anthony deangelo',
    # Goalies — contracts CSV uses full names, MoneyPuck uses nicknames
    'phillip grubauer':      'philipp grubauer',
    'samuel montembeault':   'sam montembeault',
    'daniel vladar':         'dan vladar',
    'cameron talbot':        'cam talbot',
    'matthew murray':        'matt murray',
}

def match_player(name):
    n = norm(name)
    # 0. Manual override
    if n in OVERRIDES:
        n = OVERRIDES[n]
    # 1. Exact match
    if n in name_to_id:
        return name_to_id[n], 'exact'
    # 2. Last + first initial
    parts = n.split()
    if len(parts) >= 2:
        last, init = parts[-1], parts[0][0]
        candidates = [
            (k, v) for k, v in name_to_id.items()
            if (lambda p: p and len(p) >= 2 and p[-1] == last and p[0][0] == init)(k.split())
        ]
        if len(candidates) == 1:
            return candidates[0][1], 'last+initial'
        elif len(candidates) > 1:
            return candidates[0][1], 'last+initial (ambiguous)'
    return None, 'unmatched'

contracts['mp_id']        = None
contracts['match_method'] = None

for idx, row in contracts.iterrows():
    pid, method = match_player(row['player_name'])
    contracts.at[idx, 'mp_id']        = pid
    contracts.at[idx, 'match_method'] = method

print('Match results:')
print(contracts['match_method'].value_counts())
print()
matched   = contracts[contracts['mp_id'].notna()]
unmatched = contracts[contracts['mp_id'].isna()]
print(f'Matched:   {len(matched):,} / {len(contracts):,} ({len(matched)/len(contracts)*100:.1f}%)')
print(f'Unmatched: {len(unmatched):,}')
print()
print('Unmatched contracts (review):')
print(unmatched[['player_name','position','start_year','aav']].to_string())

## 4. Fetch All MoneyPuck Stat Seasons

Fetch all four situations (all, 5on5, pp, sh) for skaters and goalies.
This cell makes ~48 HTTP requests — takes about 2–3 minutes.

In [ ]:
# Columns we want from each situation
ALL_COLS = [
    'playerId','season','name','team','position','games_played','icetime',
    'gameScore',
    'onIce_xGoalsPercentage','offIce_xGoalsPercentage',
    'onIce_corsiPercentage','offIce_corsiPercentage',
    'I_F_goals','I_F_primaryAssists','I_F_secondaryAssists',
    'I_F_xGoals','I_F_highDangerGoals','I_F_highDangerShots',
    'I_F_shotsOnGoal','I_F_rebounds',
    'I_F_hits','I_F_takeaways','I_F_giveaways','I_F_dZoneGiveaways',
    'penaltiesDrawn','penalties','shotsBlockedByPlayer',
    'faceoffsWon','faceoffsLost',
    'OnIce_F_xGoals','OnIce_A_xGoals',
    'OnIce_F_shotAttempts','OnIce_A_shotAttempts',
    'OnIce_F_highDangerShots','OnIce_A_highDangerShots',
    'I_F_oZoneShiftStarts','I_F_dZoneShiftStarts','I_F_neutralZoneShiftStarts',
]
# Actual column names from the MoneyPuck goalies CSV (goalie perspective: shots/goals faced)
GOALIE_ALL_COLS = [
    'playerId','season','name','team','position','games_played','icetime',
    'ongoal',               # shots on goal against
    'goals',                # goals against
    'xGoals',               # expected goals against
    'flurryAdjustedxGoals',
    'highDangerShots','highDangerGoals','highDangerxGoals',
    'mediumDangerShots','mediumDangerGoals','mediumDangerxGoals',
    'lowDangerShots','lowDangerGoals','lowDangerxGoals',
    'unblocked_shot_attempts',
]
SPLIT_COLS = [
    'playerId','season','situation','icetime',
    'I_F_goals','I_F_primaryAssists','I_F_secondaryAssists','I_F_xGoals',
    'OnIce_F_xGoals','OnIce_A_xGoals',
    'OnIce_F_shotAttempts','OnIce_A_shotAttempts',
]

def safe_cols(df, wanted):
    return [c for c in wanted if c in df.columns]

# Store raw frames: key = (season, ptype, situation)
# Uses cached downloads — no CSV is fetched more than once
mp_frames = {}

for season in SEASONS:
    print(f'Season {season}:', end=' ')
    for ptype in ['skaters', 'goalies']:
        for sit in [SIT_ALL, SIT_EV, SIT_PP, SIT_PK]:
            df = fetch_mp(season, ptype, sit)
            if not df.empty:
                if ptype == 'goalies' and sit == SIT_ALL:
                    df = df[safe_cols(df, GOALIE_ALL_COLS)]
                elif sit == SIT_ALL:
                    df = df[safe_cols(df, ALL_COLS)]
                else:
                    df = df[safe_cols(df, SPLIT_COLS)]
                df['season'] = season
                df['playerId'] = df['playerId'].astype(int).astype(str)
                mp_frames[(season, ptype, sit)] = df
        print(f'{ptype}+', end=' ')
    print()

print('\nAll fetches complete.')
for season in SEASONS:
    counts = {sit: len(mp_frames.get((season, 'skaters', sit), pd.DataFrame()))
              for sit in [SIT_ALL, SIT_EV, SIT_PP, SIT_PK]}
    print(f'  {season}: all={counts[SIT_ALL]}, ev={counts[SIT_EV]}, pp={counts[SIT_PP]}, pk={counts[SIT_PK]}')

## 5. Build One Row per Player-Season

For skaters: merge all + 5on5 (`ev_` prefix) + pp (`pp_` prefix) + sh (`sh_` prefix).
For goalies: all-situations only (no meaningful splits needed).

In [ ]:
def prefix_df(df, prefix):
    """Prefix all columns except playerId and season."""
    keep = {'playerId', 'season', 'situation'}
    rename = {c: f'{prefix}_{c}' for c in df.columns if c not in keep}
    return df.drop(columns=['situation'], errors='ignore').rename(columns=rename)

skater_seasons = []
for season in SEASONS:
    base = mp_frames.get((season, 'skaters', SIT_ALL), pd.DataFrame())
    if base.empty:
        continue
    merged = base.copy()
    for sit, pfx in [(SIT_EV, 'ev'), (SIT_PP, 'pp'), (SIT_PK, 'sh')]:
        split = mp_frames.get((season, 'skaters', sit), pd.DataFrame())
        if not split.empty:
            split_p = prefix_df(split, pfx)
            merged = merged.merge(split_p, on=['playerId', 'season'], how='left')
    skater_seasons.append(merged)

goalie_seasons = []
for season in SEASONS:
    g = mp_frames.get((season, 'goalies', SIT_ALL), pd.DataFrame())
    if not g.empty:
        goalie_seasons.append(g)

skaters_all_seasons = pd.concat(skater_seasons, ignore_index=True) if skater_seasons else pd.DataFrame()
goalies_all_seasons  = pd.concat(goalie_seasons, ignore_index=True) if goalie_seasons  else pd.DataFrame()

print(f'Skater rows (all seasons): {len(skaters_all_seasons):,}')
print(f'Goalie rows (all seasons): {len(goalies_all_seasons):,}')
print()
pp_cols_found = [c for c in skaters_all_seasons.columns if c.startswith('pp_')]
sh_cols_found = [c for c in skaters_all_seasons.columns if c.startswith('sh_')]
ev_cols_found = [c for c in skaters_all_seasons.columns if c.startswith('ev_')]
print(f'ev_ columns: {len(ev_cols_found)}')
print(f'pp_ columns: {len(pp_cols_found)}')
print(f'sh_ columns: {len(sh_cols_found)}')

## 6. Join Contracts → Stats

For each matched contract, look up the player's MoneyPuck row for their `stat_season`.

In [ ]:
matched_contracts = contracts[contracts['mp_id'].notna()].copy()
matched_contracts['stat_season'] = matched_contracts['stat_season'].astype(str)

# Tag goalies by position
matched_contracts['is_goalie'] = matched_contracts['position'].str.upper() == 'G'

# Split into skaters and goalies
skater_contracts = matched_contracts[~matched_contracts['is_goalie']].copy()
goalie_contracts = matched_contracts[ matched_contracts['is_goalie']].copy()

skaters_all_seasons['playerId'] = skaters_all_seasons['playerId'].astype(str)
skaters_all_seasons['season']   = skaters_all_seasons['season'].astype(str)

joined_skaters = skater_contracts.merge(
    skaters_all_seasons,
    left_on  = ['mp_id', 'stat_season'],
    right_on = ['playerId', 'season'],
    how='left',
    suffixes=('', '_mp')
)

goalies_all_seasons['playerId'] = goalies_all_seasons['playerId'].astype(str)
goalies_all_seasons['season']   = goalies_all_seasons['season'].astype(str)

joined_goalies = goalie_contracts.merge(
    goalies_all_seasons,
    left_on  = ['mp_id', 'stat_season'],
    right_on = ['playerId', 'season'],
    how='left',
    suffixes=('', '_mp')
)

combined = pd.concat([joined_skaters, joined_goalies], ignore_index=True)

print(f'Contracts with stats join result: {len(combined):,}')
print(f'  - Has MoneyPuck stats (games_played not null): {combined["games_played"].notna().sum():,}')
print(f'  - No stats found (no NHL games that season):  {combined["games_played"].isna().sum():,}')

## 7. Apply GP ≥ 40 Filter

In [ ]:
has_stats  = combined[combined['games_played'].notna()].copy()
no_stats   = combined[combined['games_played'].isna()].copy()

before_gp  = len(has_stats)

# Separate GP thresholds: skaters play 82 games, goalies share starts with backups
is_goalie_row = has_stats['is_goalie'].fillna(False)
skater_pass   = (~is_goalie_row) & (has_stats['games_played'] >= MIN_GP_SKATER)
goalie_pass   = ( is_goalie_row) & (has_stats['games_played'] >= MIN_GP_GOALIE)
has_stats     = has_stats[skater_pass | goalie_pass].copy()
after_gp      = len(has_stats)

print(f'Had stats:                    {before_gp:,}')
print(f'Passed GP filter (skater>=40, goalie>=20): {after_gp:,}  (dropped {before_gp - after_gp:,})')
print(f'No stats at all:              {len(no_stats):,}  (prospects/retired)')
print()
print(f'FINAL TRAINING ROWS: {after_gp:,}')
print()
print('By position:')
print(has_stats['position'].value_counts())

## 8. Spot Check — Known Players

In [ ]:
check_names = ['Connor McDavid','Nathan MacKinnon','Auston Matthews','Cale Makar','Sidney Crosby','Mats Zuccarello']
spot_cols = ['player_name','stat_season','position','games_played','I_F_goals','I_F_primaryAssists','aav','start_year','match_method']
existing  = [c for c in spot_cols if c in has_stats.columns]

spot = has_stats[has_stats['player_name'].isin(check_names)][existing].sort_values(['player_name','stat_season'])
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
spot

## 9. Null Audit on Key Columns

In [ ]:
audit_cols = [
    'games_played','icetime','gameScore',
    'onIce_xGoalsPercentage','offIce_xGoalsPercentage','onIce_corsiPercentage',
    'I_F_goals','I_F_xGoals','I_F_highDangerGoals',
    'I_F_hits','I_F_takeaways','I_F_giveaways','I_F_dZoneGiveaways',
    'faceoffsWon','faceoffsLost','shotsBlockedByPlayer',
    'OnIce_F_xGoals','OnIce_A_xGoals',
    'OnIce_F_shotAttempts','OnIce_A_shotAttempts',
    'OnIce_F_highDangerShots','OnIce_A_highDangerShots',
    'pp_icetime','pp_I_F_goals','pp_I_F_xGoals',
    'sh_icetime','ev_icetime',
    'aav','age_at_signing',
]
existing = [c for c in audit_cols if c in has_stats.columns]
nulls = has_stats[existing].isnull().mean().mul(100).round(1).sort_values(ascending=False)
print('Null rates (%) for key columns:')
print(nulls[nulls > 0].to_string() if nulls[nulls > 0].any() else 'No nulls — clean!')

## 10. Export

In [ ]:
out_dir  = os.path.join(os.getcwd(), 'data')
os.makedirs(out_dir, exist_ok=True)
out_path = os.path.join(out_dir, 'raw_combined.csv')

has_stats.to_csv(out_path, index=False)

print(f'Saved {len(has_stats):,} rows x {len(has_stats.columns)} columns')
print(f'→ {out_path}')
print()
print('=' * 60)
print('STOP POINT — review before notebook 02')
print('  1. Final row count (target: 700-900)')
print('  2. Null rates — any key features mostly missing?')
print('  3. Spot check — do known players look right?')
print('  4. Are pp_ and sh_ columns populated?')
print('=' * 60)